# Step 08A — Prepare exactly 11 expert-positive, source-label-negative radiographs for CVAT

**Purpose:** Prepare *unmodified* image copies for a blinded manual caries-bounding-box annotation task. This notebook **does not annotate images, train models, modify the source labels, or infer lesion locations**.

- Internal: 3 images, source boxes = 0.
- External: 8 images, source boxes = 0.
- Class name in CVAT: **caries**, rectangle only, YOLO class ID 0.
- Maintain separate tasks for internal and external images. Do not show annotators any model predictions.
- **Only export the images into CVAT if the original datasets' licensing and institutional data-use rules permit third-party cloud uploads.** Otherwise use a permitted/self-hosted platform.

After clinician approval of new boxes, export Ultralytics YOLO Detection annotations for both tasks and **do not overwrite original GT files**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from hashlib import sha256
import csv, json, os

ROOT = Path('/content/drive/MyDrive/Pediatric_Caries_AI')
INTERNAL_IMAGES = ROOT/'processed_data/yolo_caries_baseline/images/test'
EXTERNAL_IMAGES = ROOT/'external_validation/MOPG7_66/images/test'
INTERNAL_LABELS = ROOT/'processed_data/yolo_caries_baseline/labels/test'
EXTERNAL_LABELS = ROOT/'external_validation/MOPG7_66/labels/test'
OUT = ROOT/'reviewer_revision/Step_08A_CVAT_11_Expert_Positive'
OUT.mkdir(parents=True, exist_ok=True)

COHORT_IMAGES={
    'internal': ['cate1_009.png', 'cate1_010.png', 'cate2_010.png'],
    'external': ['00001247.jpg','00001373.jpg','00001381.jpg','0000538.jpg',
                 '0000623.jpg','ibn159.jpg','ibn285.jpg','ibn59.jpg']
}
assert len(COHORT_IMAGES['internal'])==3 and len(COHORT_IMAGES['external'])==8
for path in (INTERNAL_IMAGES,EXTERNAL_IMAGES):
    assert path.is_dir(), f'Image folder missing: {path}'
print('Internal images:',INTERNAL_IMAGES)
print('External images:',EXTERNAL_IMAGES)
print('Output:',OUT)


In [ ]:
# Check all 11 originals and source-label-negative status before packaging.
records=[]
for cohort, names in COHORT_IMAGES.items():
    base=INTERNAL_IMAGES if cohort=='internal' else EXTERNAL_IMAGES
    labdir=INTERNAL_LABELS if cohort=='internal' else EXTERNAL_LABELS
    for name in names:
        image=base/name
        assert image.is_file(), f'Missing: {image}'
        label=labdir/(Path(name).stem+'.txt')
        if label.exists():
            lines=[l for l in label.read_text(encoding='utf-8').splitlines() if l.strip()]
            assert len(lines)==0, f'STOP — source annotations are not empty: {label}; inspect input path before proceeding.'
            label_status='existing_empty'
        else:
            # Some YOLO pipelines represent negative images by an absent .txt label.
            label_status='absent'
        with image.open('rb') as f:
            digest=sha256(f.read()).hexdigest()
        records.append(dict(cohort=cohort, image=name, source_image_path=str(image),
                            image_sha256=digest, original_label_path=str(label),
                            original_label_status=label_status,
                            expert_image_level_decision='Yes',
                            new_box_count='PENDING CLINICIAN APPROVAL',
                            corrected_label_filename=Path(name).stem+'.txt'))
assert len(records)==11
print('PASS: 11 original image files found and all 11 have zero original caries label rows')
for r in records:print(r['cohort'],r['image'],r['original_label_status'])


In [ ]:
# Create FLAT image ZIPs for upload as two separate CVAT tasks.
zip_specs=[('internal','CVAT_Internal_3_Raw_Images.zip'),
           ('external','CVAT_External_8_Raw_Images.zip')]
for cohort,filename in zip_specs:
    zpath=OUT/filename
    base=INTERNAL_IMAGES if cohort=='internal' else EXTERNAL_IMAGES
    with ZipFile(zpath,'w',ZIP_DEFLATED,compresslevel=6) as z:
        for name in COHORT_IMAGES[cohort]:
            z.write(base/name,arcname=name)
    with ZipFile(zpath) as z:
        got=sorted(z.namelist())
        assert got==sorted(COHORT_IMAGES[cohort]),(filename,got)
    print('Created:',zpath, 'bytes:',zpath.stat().st_size)

manifest=OUT/'CVAT_11_Image_Source_Manifest.csv'
with manifest.open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
readme=OUT/'READ_ME_CVAT_11.txt'
readme.write_text('''CVAT ANNOTATION WORK — 11 EXPERT-POSITIVE SOURCE-NEGATIVE IMAGES

Create one CVAT project with ONE label named caries (shape: Rectangle).
Create TWO separate tasks within this project:
Task A: 3 internal images from CVAT_Internal_3_Raw_Images.zip
Task B: 8 external images from CVAT_External_8_Raw_Images.zip

Only draw a bounding box when the relevant carious lesion is identifiable under the agreed study definition.
Image-level expert Yes does NOT specify the lesion's location or number.
If no defensible target can be located, leave it undocumented as a box and flag for clinician review.

Save each CVAT job. Export each task in Ultralytics YOLO Detection 1.0 format (without images if restricted).
Store exported ZIP archives separately and keep all original image/label files unchanged.
A dental expert must review and approve every proposed corrected box BEFORE any revised-ground-truth analysis.
Do NOT use model-generated predictions to decide how to draw boxes.
For missing annotations on these 11 originally negative images, each exported per-image txt should contain
all clinically approved target boxes (one line per lesion; class id 0, normalized cx cy width height).
Task folders should NOT be mixed because the internal/external distinction matters for evaluation.
''',encoding='utf-8')
print('Manifest:',manifest)
print('Instructions:',readme)
print('\nDONE. Download both image ZIPs from Google Drive, upload each as one CVAT task, then send the CVAT annotation export files back for validation.')
